In [1]:
import os
import numpy as np
import torch
from torchvision import transforms
from torchvision.transforms import InterpolationMode  # 明确导入插值模式
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F


# --------------------------
# 1. 生成器结构（与训练代码100%一致，不可修改）
# --------------------------
class Generator(nn.Module):
    def __init__(self, input_channels=1, output_channels=3):
        super(Generator, self).__init__()

        # 编码器（与训练一致）
        self.enc1 = nn.Sequential(
            nn.Conv2d(input_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # 调色板处理（输入3*6=18，适配(6,3)维度）
        self.palette_fc = nn.Sequential(
            nn.Linear(3 * 6, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True)
        )

        # 解码器（与训练一致，含跳跃连接）
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(512 + 512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(256 + 256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(128 + 128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(64 + 64, output_channels, 4, 2, 1),
            nn.Tanh()  # 输出[-1,1]，与训练一致
        )

    def forward(self, line, palette):
        e1 = self.enc1(line)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        batch_size = palette.size(0)
        palette_flat = palette.view(batch_size, -1)  # (1,6,3)→(1,18)，与训练一致
        p_feat = self.palette_fc(palette_flat)
        p_feat = p_feat.view(batch_size, 512, 1, 1)
        p_feat = F.interpolate(p_feat, size=e4.size()[2:], mode='nearest')

        d1 = self.dec1(torch.cat([e4, p_feat], 1))
        d2 = self.dec2(torch.cat([d1, e3], 1))
        d3 = self.dec3(torch.cat([d2, e2], 1))
        output = self.dec4(torch.cat([d3, e1], 1))

        return output


# --------------------------
# 2. 模型加载（兼容多GPU训练权重，强制eval模式）
# --------------------------
def load_generator(model_path, device):
    """
    加载生成器，自动处理多GPU权重（去除module.前缀），强制进入评估模式
    """
    # 初始化生成器（与训练一致）
    generator = Generator().to(device)

    # 加载权重（兼容单/多GPU训练结果）
    try:
        state_dict = torch.load(model_path, map_location=device)
        # 处理多GPU训练的权重（键名含module.）
        if any(key.startswith('module.') for key in state_dict.keys()):
            print(f"🔧 检测到多GPU训练权重，自动去除module.前缀")
            state_dict = {key.replace('module.', ''): val for key, val in state_dict.items()}

        # 加载权重并验证
        generator.load_state_dict(state_dict, strict=True)  # strict=True确保权重完全匹配
    except RuntimeError as e:
        raise RuntimeError(f"❌ 权重加载失败！生成器结构与训练时不一致。错误：{str(e)}")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"❌ 权重文件不存在：{model_path}")

    # 强制切换到评估模式（关键！关闭BatchNorm实时统计）
    generator.eval()
    print(f"✅ 模型加载成功！权重路径：{model_path}，设备：{device}，模式：eval")
    return generator


# --------------------------
# 3. 输入预处理（与训练100%对齐，含详细日志）
# --------------------------
def preprocess_single(line_img_path, palette_npy_path, device, verbose=False):
    """
    预处理单组输入：线条图（单通道）+ 调色板（6,3）
    与训练时的transform完全一致，避免效果差异
    """
    # --------------------------
    # 线条图预处理（与训练transform['line']完全一致）
    # --------------------------
    line_transform = transforms.Compose([
        # 明确插值方式（训练默认BILINEAR，避免推理用其他插值）
        transforms.Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),  # 转为[0,1]，(H,W)→(1,H,W)
        transforms.Normalize((0.5,), (0.5,))  # 单通道归一化：(x-0.5)/0.5 → [-1,1]
    ])

    # 加载并处理线条图
    try:
        line_img = Image.open(line_img_path).convert('L')  # 转为单通道灰度图（与训练一致）
        line_tensor = line_transform(line_img).unsqueeze(0).to(device)  # 加批次维度→(1,1,256,256)
        if verbose:
            print(f"📊 线条图处理完成：路径={line_img_path}，尺寸={line_tensor.shape}，范围=[{line_tensor.min():.2f},{line_tensor.max():.2f}]")
    except Exception as e:
        raise ValueError(f"❌ 线条图处理失败！路径：{line_img_path}，错误：{str(e)}")

    # --------------------------
    # 调色板预处理（与训练一致，适配(6,3)维度）
    # --------------------------
    try:
        palette = np.load(palette_npy_path)
        # 校验维度（必须是(6,3)，与训练一致）
        if palette.shape != (6, 3):
            raise ValueError(f"维度应为(6,3)，实际为{palette.shape}")
        # 归一化到[0,1]（与训练一致）
        palette_tensor = torch.from_numpy(palette).float() / 255.0
        # 加批次维度→(1,6,3)
        palette_tensor = palette_tensor.unsqueeze(0).to(device)
        if verbose:
            print(f"📊 调色板处理完成：路径={palette_npy_path}，尺寸={palette_tensor.shape}，范围=[{palette_tensor.min():.2f},{palette_tensor.max():.2f}]")
    except Exception as e:
        raise ValueError(f"❌ 调色板处理失败！路径：{palette_npy_path}，错误：{str(e)}")

    return line_tensor, palette_tensor, line_img  # 返回原始线条图用于可视化（可选）


# --------------------------
# 4. 批量生成（强制eval模式，与训练后处理一致）
# --------------------------
def batch_generate(generator, line_dir, palette_dir, output_dir, device, verbose=False):
    """
    批量生成彩色图像，全程强制模型在eval模式，后处理与训练一致
    """
    # 强制确保模型在评估模式（防止意外切换到train）
    generator.eval()
    print(f"📌 批量生成开始，模型模式：{('train' if generator.training else 'eval')}（必须是eval）")

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 生成结果保存目录：{output_dir}")

    # 获取有效线条图文件（过滤隐藏文件和非图像格式）
    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    line_files = [f for f in os.listdir(line_dir)
                  if f.lower().endswith(valid_ext)
                  and not f.startswith('.')]  # 排除.gitignore、.DS_Store等

    total_files = len(line_files)
    if total_files == 0:
        print("⚠️  未在线条图目录找到有效文件！目录：{line_dir}")
        return
    print(f"🔍 发现{total_files}个有效线条图，开始批量生成...")

    # 批量处理
    success_count = 0
    for idx, filename in enumerate(line_files, 1):
        base_name = os.path.splitext(filename)[0]
        # 构建文件路径（确保线条图与调色板同名匹配）
        line_path = os.path.join(line_dir, filename)
        palette_path = os.path.join(palette_dir, f"{base_name}.npy")

        # 跳过缺失调色板的文件
        if not os.path.exists(palette_path):
            print(f"⚠️  [{idx}/{total_files}] 跳过：{filename} → 未找到调色板{palette_path}")
            continue

        try:
            # 1. 预处理输入（与训练一致）
            line_tensor, palette_tensor, line_img = preprocess_single(
                line_path, palette_path, device, verbose=verbose
            )

            # 2. 推理生成（关闭梯度，加速+省内存）
            with torch.no_grad():
                # 强制禁用随机操作（如Dropout，虽然生成器没有，但防后续修改）
                torch.manual_seed(42)  # 固定随机种子，确保结果可复现
                generated_tensor = generator(line_tensor, palette_tensor)

            # 3. 后处理（与训练保存逻辑完全一致，避免像素值错误）
            # 步骤：[-1,1] → [0,1] → [0,255] → 通道转置(C,H,W)→(H,W,C) → 转uint8
            generated_img = generated_tensor.squeeze().cpu()  # 去除批次维度→(3,256,256)
            generated_img = (generated_img * 0.5 + 0.5)  # 转为[0,1]（与训练一致）
            generated_img = (generated_img * 255.0).clamp(0, 255)  # 限制范围，避免溢出
            generated_img = generated_img.permute(1, 2, 0).numpy().astype(np.uint8)  # 通道转置

            # 4. 保存生成图（与线条图同名，保持原格式）
            save_path = os.path.join(output_dir, filename)
            Image.fromarray(generated_img).save(save_path)

            success_count += 1
            # 打印进度（每10个或最后一个文件）
            if idx % 10 == 0 or idx == total_files:
                print(f"✅ [{idx}/{total_files}] 生成成功：{save_path}")

        except Exception as e:
            print(f"❌ [{idx}/{total_files}] 处理失败：{filename} → 错误：{str(e)}")
            continue

    # 生成总结
    print(f"\n🎉 批量生成结束！")
    print(f"📊 统计：总文件数{total_files} | 成功{success_count} | 失败{total_files - success_count}")
    print(f"📁 结果目录：{output_dir}")


# --------------------------
# 5. 主执行流程（用户仅需修改路径）
# --------------------------
if __name__ == "__main__":
    # 1. 自动选择设备（优先GPU，无GPU则CPU）
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"💻 设备信息：{device}")
    if device.type == 'cuda':
        print(f"   GPU型号：{torch.cuda.get_device_name(0)}，显存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}GB")

    # 2. 用户配置（仅需修改这4个路径！）
    CONFIG = {
        "model_path": "generator_final_dc_hed_contour.pth",  # 训练后保存的权重文件（如checkpoint03/generator_final_hed_contour_new.pth）
        "line_dir": "../dataset/line_drawing_dc_hed_contour",  # 待生成的线条图目录
        "palette_dir": "../dataset/color_palette",  # 调色板目录（(6,3)维度，与线条图同名）
        "output_dir": "generated_results_dc_hed_contour",  # 生成图保存目录（自动创建）
        "verbose": False  # 是否打印详细预处理日志（调试用，默认关闭）
    }

    # 3. 加载模型
    print(f"\n📥 正在加载模型：{CONFIG['model_path']}")
    try:
        generator = load_generator(CONFIG['model_path'], device)
    except Exception as e:
        print(f"❌ 模型加载失败，程序退出：{str(e)}")
        exit(1)

    # 4. 批量生成
    print(f"\n🚀 开始批量生成（线条图目录：{CONFIG['line_dir']}）")
    try:
        batch_generate(
            generator=generator,
            line_dir=CONFIG['line_dir'],
            palette_dir=CONFIG['palette_dir'],
            output_dir=CONFIG['output_dir'],
            device=device,
            verbose=CONFIG['verbose']
        )
    except Exception as e:
        print(f"❌ 批量生成异常：{str(e)}")
        exit(1)

💻 设备信息：cuda
   GPU型号：NVIDIA GeForce RTX 4060 Laptop GPU，显存：8.00GB

📥 正在加载模型：generator_final_dc_hed_contour.pth
✅ 模型加载成功！权重路径：generator_final_dc_hed_contour.pth，设备：cuda，模式：eval

🚀 开始批量生成（线条图目录：../dataset/line_drawing_dc_hed_contour）
📌 批量生成开始，模型模式：eval（必须是eval）
📁 生成结果保存目录：generated_results_dc_hed_contour
🔍 发现10076个有效线条图，开始批量生成...
✅ [10/10076] 生成成功：generated_results_dc_hed_contour\000010.jpg
✅ [20/10076] 生成成功：generated_results_dc_hed_contour\000020.jpg
✅ [30/10076] 生成成功：generated_results_dc_hed_contour\000030.jpg
✅ [40/10076] 生成成功：generated_results_dc_hed_contour\000040.jpg
✅ [50/10076] 生成成功：generated_results_dc_hed_contour\000050.jpg
✅ [60/10076] 生成成功：generated_results_dc_hed_contour\000060.jpg
✅ [70/10076] 生成成功：generated_results_dc_hed_contour\000070.jpg
✅ [80/10076] 生成成功：generated_results_dc_hed_contour\000080.jpg
✅ [90/10076] 生成成功：generated_results_dc_hed_contour\000090.jpg
✅ [100/10076] 生成成功：generated_results_dc_hed_contour\000100.jpg
✅ [110/10076] 生成成功：generated_results_dc_hed_conto

In [2]:
import os
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F


# --------------------------
# 1. 生成器结构（与新训练代码100%一致）
# --------------------------
class Generator(nn.Module):
    def __init__(self, input_channels=1, output_channels=3):
        super(Generator, self).__init__()

        # 编码器部分
        self.enc1 = nn.Sequential(
            nn.Conv2d(input_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # 调色板特征处理
        self.palette_fc = nn.Sequential(
            nn.Linear(3 * 6, 256),  # 6色调色板
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True)
        )

        # 解码器部分
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(512 + 512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(256 + 256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(128 + 128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(64 + 64, output_channels, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, line, palette):
        # 编码器前向传播
        e1 = self.enc1(line)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # 处理调色板特征
        batch_size = palette.size(0)
        palette_flat = palette.view(batch_size, -1)
        p_feat = self.palette_fc(palette_flat)
        p_feat = p_feat.view(batch_size, 512, 1, 1)
        p_feat = F.interpolate(p_feat, size=e4.size()[2:], mode='nearest')

        # 解码器前向传播（带跳跃连接）
        d1 = self.dec1(torch.cat([e4, p_feat], 1))
        d2 = self.dec2(torch.cat([d1, e3], 1))
        d3 = self.dec3(torch.cat([d2, e2], 1))
        output = self.dec4(torch.cat([d3, e1], 1))

        return output


# --------------------------
# 2. 模型加载（兼容消融实验权重）
# --------------------------
def load_generator(model_path, device):
    """
    加载生成器，自动处理多GPU训练权重，强制eval模式
    """
    # 初始化生成器（与新训练代码一致）
    generator = Generator().to(device)

    # 加载权重（兼容单/多GPU训练结果）
    try:
        state_dict = torch.load(model_path, map_location=device)
        # 处理多GPU训练的权重（键名含module.）
        if any(key.startswith('module.') for key in state_dict.keys()):
            print(f"🔧 检测到多GPU训练权重，自动去除module.前缀")
            state_dict = {key.replace('module.', ''): val for key, val in state_dict.items()}

        # 加载权重并验证
        generator.load_state_dict(state_dict, strict=True)
    except RuntimeError as e:
        raise RuntimeError(f"❌ 权重加载失败！生成器结构与训练时不一致。错误：{str(e)}")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"❌ 权重文件不存在：{model_path}")

    # 强制切换到评估模式
    generator.eval()
    print(f"✅ 模型加载成功！权重路径：{model_path}，设备：{device}，模式：eval")
    return generator


# --------------------------
# 3. 输入预处理（与新训练代码100%对齐）
# --------------------------
def preprocess_single(line_img_path, palette_npy_path, device, verbose=False):
    """
    预处理单组输入：线条图+调色板，与新训练代码的transform完全一致
    """
    # --------------------------
    # 线条图预处理（与训练transform['line']完全一致）
    # --------------------------
    line_transform = transforms.Compose([
        transforms.Resize((256, 256)),  # 与训练代码一致，使用默认插值
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 加载并处理线条图
    try:
        line_img = Image.open(line_img_path).convert('L')
        line_tensor = line_transform(line_img).unsqueeze(0).to(device)
        if verbose:
            print(f"📊 线条图处理完成：路径={line_img_path}，尺寸={line_tensor.shape}，范围=[{line_tensor.min():.2f},{line_tensor.max():.2f}]")
    except Exception as e:
        raise ValueError(f"❌ 线条图处理失败！路径：{line_img_path}，错误：{str(e)}")

    # --------------------------
    # 调色板预处理（与训练一致）
    # --------------------------
    try:
        palette = np.load(palette_npy_path)
        if palette.shape != (6, 3):
            raise ValueError(f"维度应为(6,3)，实际为{palette.shape}")
        palette_tensor = torch.from_numpy(palette).float() / 255.0
        palette_tensor = palette_tensor.unsqueeze(0).to(device)
        if verbose:
            print(f"📊 调色板处理完成：路径={palette_npy_path}，尺寸={palette_tensor.shape}，范围=[{palette_tensor.min():.2f},{palette_tensor.max():.2f}]")
    except Exception as e:
        raise ValueError(f"❌ 调色板处理失败！路径：{palette_npy_path}，错误：{str(e)}")

    return line_tensor, palette_tensor, line_img


# --------------------------
# 4. 批量生成（适配消融实验结果）
# --------------------------
def batch_generate(generator, line_dir, palette_dir, output_dir, device, verbose=False):
    """
    批量生成彩色图像，全程强制eval模式，后处理与训练一致
    """
    generator.eval()
    print(f"📌 批量生成开始，模型模式：{('train' if generator.training else 'eval')}")

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 生成结果保存目录：{output_dir}")

    # 获取有效线条图文件
    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    line_files = [f for f in os.listdir(line_dir)
                  if f.lower().endswith(valid_ext)
                  and not f.startswith('.')]

    total_files = len(line_files)
    if total_files == 0:
        print("⚠️  未在线条图目录找到有效文件！目录：{line_dir}")
        return
    print(f"🔍 发现{total_files}个有效线条图，开始批量生成...")

    # 批量处理
    success_count = 0
    for idx, filename in enumerate(line_files, 1):
        base_name = os.path.splitext(filename)[0]
        line_path = os.path.join(line_dir, filename)
        palette_path = os.path.join(palette_dir, f"{base_name}.npy")

        if not os.path.exists(palette_path):
            print(f"⚠️  [{idx}/{total_files}] 跳过：{filename} → 未找到调色板{palette_path}")
            continue

        try:
            # 预处理输入
            line_tensor, palette_tensor, line_img = preprocess_single(
                line_path, palette_path, device, verbose=verbose
            )

            # 推理生成
            with torch.no_grad():
                torch.manual_seed(42)
                generated_tensor = generator(line_tensor, palette_tensor)

            # 后处理（与训练保存逻辑一致）
            generated_img = generated_tensor.squeeze().cpu()
            generated_img = (generated_img * 0.5 + 0.5)
            generated_img = (generated_img * 255.0).clamp(0, 255)
            generated_img = generated_img.permute(1, 2, 0).numpy().astype(np.uint8)

            # 保存生成图
            save_path = os.path.join(output_dir, filename)
            Image.fromarray(generated_img).save(save_path)

            success_count += 1
            if idx % 10 == 0 or idx == total_files:
                print(f"✅ [{idx}/{total_files}] 生成成功：{save_path}")

        except Exception as e:
            print(f"❌ [{idx}/{total_files}] 处理失败：{filename} → 错误：{str(e)}")
            continue

    # 生成总结
    print(f"\n🎉 批量生成结束！")
    print(f"📊 统计：总文件数{total_files} | 成功{success_count} | 失败{total_files - success_count}")
    print(f"📁 结果目录：{output_dir}")


# --------------------------
# 5. 单张图片可视化生成
# --------------------------
def generate_single_visualization(generator, line_img_path, palette_npy_path, save_path, device):
    """
    生成单张图片并可视化对比（线条图+生成图）
    """
    # 预处理
    line_tensor, palette_tensor, line_img = preprocess_single(line_img_path, palette_npy_path, device, verbose=True)

    # 生成
    with torch.no_grad():
        generated_tensor = generator(line_tensor, palette_tensor)

    # 后处理
    generated_img = generated_tensor.squeeze().cpu()
    generated_img = (generated_img * 0.5 + 0.5).clamp(0, 1)
    generated_img = generated_img.permute(1, 2, 0).numpy()

    # 可视化
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(line_img, cmap='gray')
    plt.title('Line Drawing', fontsize=14)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(generated_img)
    plt.title('Generated Image', fontsize=14)
    plt.axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"🎨 单张可视化结果已保存：{save_path}")


# --------------------------
# 6. 主执行流程（适配消融实验配置）
# --------------------------
if __name__ == "__main__":
    # 1. 自动选择设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"💻 设备信息：{device}")
    if device.type == 'cuda':
        print(f"   GPU型号：{torch.cuda.get_device_name(0)}，显存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}GB")

    # 2. 用户配置（适配消融实验路径）
    CONFIG = {
        # 消融实验权重路径示例（根据实际情况修改）
        "model_path": "generator_final_dc_hed_contour.pth",
        "line_dir": "../dataset/testdata_dc_hed_contour",  # 与训练一致的线条图目录
        "palette_dir": "../dataset/testdata_palette",            # 与训练一致的调色板目录
        "output_dir": "testdata_results", # 消融实验结果保存目录

        "verbose": False
    }
    #01：移除颜色感知损失
    #02：移除SSIM损失
    #03：移除风格损失
    #04：移除感知损失
    #06：将L1损失替换为MSE损失

    # 3. 加载模型
    print(f"\n📥 正在加载模型：{CONFIG['model_path']}")
    try:
        generator = load_generator(CONFIG['model_path'], device)
    except Exception as e:
        print(f"❌ 模型加载失败，程序退出：{str(e)}")
        exit(1)

    # 4. 单张图片测试（可选）

    # 5. 批量生成
    print(f"\n🚀 开始批量生成（线条图目录：{CONFIG['line_dir']}）")
    try:
        batch_generate(
            generator=generator,
            line_dir=CONFIG['line_dir'],
            palette_dir=CONFIG['palette_dir'],
            output_dir=CONFIG['output_dir'],
            device=device,
            verbose=CONFIG['verbose']
        )
    except Exception as e:
        print(f"❌ 批量生成异常：{str(e)}")
        exit(1)

💻 设备信息：cuda
   GPU型号：NVIDIA GeForce RTX 4060 Laptop GPU，显存：8.00GB

📥 正在加载模型：generator_final_dc_hed_contour.pth
✅ 模型加载成功！权重路径：generator_final_dc_hed_contour.pth，设备：cuda，模式：eval

🚀 开始批量生成（线条图目录：../dataset/testdata_dc_hed_contour）
📌 批量生成开始，模型模式：eval
📁 生成结果保存目录：testdata_results
🔍 发现20个有效线条图，开始批量生成...
✅ [10/20] 生成成功：testdata_results\214699767.jpg
✅ [20/20] 生成成功：testdata_results\215683017.jpg

🎉 批量生成结束！
📊 统计：总文件数20 | 成功20 | 失败0
📁 结果目录：testdata_results


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.gridspec as gridspec
from pathlib import Path

# 设置中文字体，确保中文正常显示
plt.rcParams["font.family"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False  # 解决负号显示问题

def create_color_palette_image(palette, size=(100, 20)):
    """将(6,3)的调色板数组转换为可视化图像"""
    # 确保调色板值在0-1范围
    if palette.max() > 1:
        palette = palette / 255.0

    img = np.zeros((size[1], size[0], 3))
    segment_width = size[0] // 6

    for i in range(6):
        start = i * segment_width
        end = (i + 1) * segment_width if i < 5 else size[0]
        img[:, start:end, :] = palette[i]

    return img

def process_and_save_images(folder1, folder2, folder3, folder4, save_dir, figsize=(16, 4)):
    """
    处理并保存图片和调色板，使用constrained_layout替代tight_layout以消除警告

    参数:
    folder1: 第一个图片文件夹路径
    folder2: 调色板npy文件文件夹路径
    folder3: 第三个图片文件夹路径
    folder4: 第四个图片文件夹路径
    save_dir: 保存结果的文件夹路径
    figsize: 画布大小
    """
    # 创建保存目录
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    # 获取所有文件并按名称排序
    def get_sorted_files(folder, extensions):
        files = [f for f in os.listdir(folder) if any(f.endswith(ext) for ext in extensions)]
        files.sort()  # 按文件名排序
        return [os.path.join(folder, f) for f in files]

    # 获取各个文件夹的文件列表
    img_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.gif']
    folder1_files = get_sorted_files(folder1, img_extensions)
    folder2_files = get_sorted_files(folder2, ['.npy'])  # 调色板文件
    folder3_files = get_sorted_files(folder3, img_extensions)
    folder4_files = get_sorted_files(folder4, img_extensions)

    # 检查文件数量是否一致
    n = len(folder1_files)
    if len(folder2_files) != n or len(folder3_files) != n or len(folder4_files) != n:
        raise ValueError(f"文件夹中的文件数量不一致: 文件夹1有{n}个, 文件夹2有{len(folder2_files)}个, "
                         f"文件夹3有{len(folder3_files)}个, 文件夹4有{len(folder4_files)}个")

    print(f"发现{n}组文件，开始处理...")

    # 处理每组文件
    for i in range(n):
        try:
            # 创建画布，使用constrained_layout替代tight_layout
            fig = plt.figure(figsize=figsize, constrained_layout=True)
            gs = gridspec.GridSpec(1, 4, figure=fig)  # 直接关联到figure

            # 显示文件夹1的图片
            ax1 = fig.add_subplot(gs[0, 0])
            img1 = Image.open(folder1_files[i]).convert('RGB')
            ax1.imshow(img1)
            ax1.set_title(f'文件夹1\n{os.path.basename(folder1_files[i])}')
            ax1.axis('off')

            # 显示文件夹2的调色板
            ax2 = fig.add_subplot(gs[0, 1])
            palette = np.load(folder2_files[i])
            if palette.shape != (6, 3):
                raise ValueError(f"调色板文件{folder2_files[i]}形状不正确，应为(6,3)，实际为{palette.shape}")

            palette_img = create_color_palette_image(palette)
            ax2.imshow(palette_img)
            ax2.set_title(f'文件夹2\n{os.path.basename(folder2_files[i])}')
            ax2.axis('off')

            # 显示文件夹3的图片
            ax3 = fig.add_subplot(gs[0, 2])
            img3 = Image.open(folder3_files[i]).convert('RGB')
            ax3.imshow(img3)
            ax3.set_title(f'文件夹3\n{os.path.basename(folder3_files[i])}')
            ax3.axis('off')

            # 显示文件夹4的图片
            ax4 = fig.add_subplot(gs[0, 3])
            img4 = Image.open(folder4_files[i]).convert('RGB')
            ax4.imshow(img4)
            ax4.set_title(f'文件夹4\n{os.path.basename(folder4_files[i])}')
            ax4.axis('off')

            # 保存图片 - 添加bbox_inches参数确保完整保存
            save_path = os.path.join(save_dir, f'{i:06d}.jpg')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()

            if (i + 1) % 10 == 0:
                print(f"已处理{i + 1}/{n}组文件")

        except Exception as e:
            print(f"处理第{i}组文件时出错: {str(e)}")
            continue

    print(f"处理完成，共生成{n}个文件，保存至: {save_dir}")

# 使用示例
if __name__ == "__main__":
    # 请根据实际情况修改以下路径
    folder1_path = "../dataset/line_drawing"  # 存放图片的文件夹1
    folder2_path = "../dataset/color_palette"  # 存放调色板npy文件的文件夹2
    folder3_path = "generated_results_hed"  # 存放图片的文件夹3
    folder4_path = "../dataset/groundtruth"  # 存放图片的文件夹4
    save_dir_path = "subplot_hed"  # 保存结果的文件夹

    # 调用函数处理并保存
    process_and_save_images(
        folder1=folder1_path,
        folder2=folder2_path,
        folder3=folder3_path,
        folder4=folder4_path,
        save_dir=save_dir_path,
        figsize=(16, 4)  # 可以根据需要调整画布大小
    )


发现10076组文件，开始处理...
已处理10/10076组文件
已处理20/10076组文件
